In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')

!rm -rf /content/graduation-p
!git clone https://github.com/berk0vic/graduation-p.git /content/graduation-p

os.chdir('/content/graduation-p/notebooks')
sys.path.insert(0, '/content/graduation-p')

!rm -rf /content/graduation-p/data
!ln -s /content/drive/MyDrive/graduation-p/data /content/graduation-p/data

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html
!pip install -q xgboost shap

# 06 — Ablation Study

**Goal:** Show what each component adds to fraud detection.

We compare 5 models on the same IEEE-CIS test set:

| Model | What it uses |
|-------|-------------|
| XGBoost | Only tabular features |
| GCN Baseline | Homogeneous graph (no attention) |
| GAT-only | Heterogeneous graph + attention (no VAE) |
| VAE-only | Reconstruction error as anomaly score (no graph) |
| **Hybrid GAT+VAE** | **Graph + VAE combined** |

If our hybrid model is better, we can see exactly which component helps and why.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from torch_geometric.loader import NeighborLoader
from sklearn.model_selection import train_test_split

from src.evaluation.metrics import compute_metrics, print_report
from src.training.losses import focal_loss

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load Graph + Split Data

In [ ]:
from src.graph.builder import build_hetero_graph

# Load raw data and build graph
train_txn = pd.read_csv('../data/raw/ieee_cis/train_transaction.csv')
train_id = pd.read_csv('../data/raw/ieee_cis/train_identity.csv')
df = train_txn.merge(train_id, on='TransactionID', how='left')
print(f'Total transactions: {len(df):,}')
print(f'Fraud rate: {df["isFraud"].mean():.3%}')

data = build_hetero_graph(df, dataset='ieee_cis')

# Labels and raw features
y = data['transaction'].y
raw_txn_features = data['transaction'].x
n_total = len(y)
print(f'\nGraph ready: {n_total:,} transactions, {raw_txn_features.shape[1]} features')

In [ ]:
# Stratified train/val/test split (70/15/15)
y_np = y.numpy()
indices = np.arange(n_total)

train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=y_np, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y_np[temp_idx], random_state=42)

# Boolean masks
train_mask = torch.zeros(n_total, dtype=torch.bool)
val_mask = torch.zeros(n_total, dtype=torch.bool)
test_mask = torch.zeros(n_total, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# Numpy arrays for tabular models
X_train = raw_txn_features[train_mask].numpy()
X_val = raw_txn_features[val_mask].numpy()
X_test = raw_txn_features[test_mask].numpy()
y_train = y[train_mask].numpy()
y_val = y[val_mask].numpy()
y_test = y[test_mask].numpy()

print(f'Train: {train_mask.sum():,} | Val: {val_mask.sum():,} | Test: {test_mask.sum():,}')
print(f'Fraud in test: {y_test.sum():,} ({y_test.mean():.3%})')

## 2. Model A — XGBoost (Tabular Baseline)

In [ ]:
from xgboost import XGBClassifier

fraud_ratio = y_train.sum() / (len(y_train) - y_train.sum())

t0 = time.time()
xgb = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    scale_pos_weight=1/fraud_ratio,
    eval_metric='aucpr', early_stopping_rounds=20,
    tree_method='hist', n_jobs=-1, random_state=42, verbosity=0,
)
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_time = time.time() - t0

xgb_scores = xgb.predict_proba(X_test)[:, 1]
xgb_metrics = compute_metrics(y_test, xgb_scores)
print(f'XGBoost trained in {xgb_time:.1f}s')
print_report(y_test, xgb_scores)

## 3. Model B — GCN Baseline (Homogeneous Graph)

In [ ]:
import torch.nn.functional as F
from src.models.baselines import GCNBaseline
from collections import defaultdict

# Build homogeneous edges: transactions sharing the same account
acct_txn_edges = data['account', 'initiates', 'transaction'].edge_index
txn_x = data['transaction'].x

acct_to_txns = defaultdict(list)
for a, t in zip(acct_txn_edges[0].tolist(), acct_txn_edges[1].tolist()):
    acct_to_txns[a].append(t)

src_list, dst_list = [], []
for txns in acct_to_txns.values():
    if len(txns) > 1:
        for i in range(len(txns)):
            for j in range(i+1, min(i+5, len(txns))):
                src_list.extend([txns[i], txns[j]])
                dst_list.extend([txns[j], txns[i]])

homo_edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
print(f'Homogeneous graph: {txn_x.shape[0]:,} nodes, {homo_edge_index.shape[1]:,} edges')

# Train GCN
gcn = GCNBaseline(in_channels=txn_x.shape[1], hidden=64, out=32).to(device)
opt_gcn = torch.optim.Adam(gcn.parameters(), lr=0.001, weight_decay=1e-4)

txn_x_dev = txn_x.to(device)
homo_edges_dev = homo_edge_index.to(device)
y_dev = y.float().to(device)
train_mask_dev = train_mask.to(device)
val_mask_dev = val_mask.to(device)

best_val = float('inf')
patience_cnt = 0

t0 = time.time()
for epoch in range(1, 101):
    gcn.train()
    opt_gcn.zero_grad()
    logits = gcn(txn_x_dev, homo_edges_dev)
    loss = focal_loss(logits[train_mask_dev], y_dev[train_mask_dev])
    loss.backward()
    opt_gcn.step()

    gcn.eval()
    with torch.no_grad():
        vl = focal_loss(gcn(txn_x_dev, homo_edges_dev)[val_mask_dev], y_dev[val_mask_dev]).item()
    if vl < best_val:
        best_val = vl; patience_cnt = 0
        best_gcn = {k: v.cpu().clone() for k, v in gcn.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= 15:
            print(f'Early stopping at epoch {epoch}')
            break
    if epoch % 25 == 0:
        print(f'  Epoch {epoch:3d} | train: {loss.item():.4f} | val: {vl:.4f}')

gcn.load_state_dict(best_gcn)
gcn_time = time.time() - t0

gcn.eval()
with torch.no_grad():
    gcn_scores_all = torch.sigmoid(gcn(txn_x_dev, homo_edges_dev)).cpu().numpy()
gcn_scores = gcn_scores_all[test_mask.numpy()]

gcn_metrics = compute_metrics(y_test, gcn_scores)
print(f'\nGCN trained in {gcn_time:.1f}s')
print_report(y_test, gcn_scores)

## 4. Model C — GAT-only (Graph Attention, No VAE)

This uses the same heterogeneous graph as our hybrid model, but **without the VAE branch**.  
If it's better than GCN → attention mechanism helps.  
If hybrid is better than this → VAE adds something extra.

In [ ]:
from src.models.baselines import GATOnlyBaseline

in_channels = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}

gat_only = GATOnlyBaseline(
    metadata=data.metadata(),
    in_channels=in_channels,
    hidden=64, out=32, heads=4,
).to(device)

opt_gat = torch.optim.Adam(gat_only.parameters(), lr=0.001, weight_decay=1e-4)

# Move data to device
x_dict_dev = {nt: data[nt].x.to(device) for nt in data.node_types}
edge_dict_dev = {et: data[et].edge_index.to(device) for et in data.edge_types}

best_val = float('inf')
patience_cnt = 0

t0 = time.time()
for epoch in range(1, 101):
    gat_only.train()
    opt_gat.zero_grad()
    logits = gat_only(x_dict_dev, edge_dict_dev)
    loss = focal_loss(logits[train_mask_dev], y_dev[train_mask_dev])
    loss.backward()
    opt_gat.step()

    gat_only.eval()
    with torch.no_grad():
        vl = focal_loss(gat_only(x_dict_dev, edge_dict_dev)[val_mask_dev], y_dev[val_mask_dev]).item()
    if vl < best_val:
        best_val = vl; patience_cnt = 0
        best_gat = {k: v.cpu().clone() for k, v in gat_only.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= 15:
            print(f'Early stopping at epoch {epoch}')
            break
    if epoch % 25 == 0:
        print(f'  Epoch {epoch:3d} | train: {loss.item():.4f} | val: {vl:.4f}')

gat_only.load_state_dict(best_gat)
gat_time = time.time() - t0

gat_only.eval()
with torch.no_grad():
    gat_scores_all = torch.sigmoid(gat_only(x_dict_dev, edge_dict_dev)).cpu().numpy()
gat_scores = gat_scores_all[test_mask.numpy()]

gat_metrics = compute_metrics(y_test, gat_scores)
print(f'\nGAT-only trained in {gat_time:.1f}s')
print_report(y_test, gat_scores)

## 5. Model D — VAE-only (Anomaly Detection, No Graph)

This uses reconstruction error as the fraud score.  
Trained on normal transactions only — fraud should have higher reconstruction error.

In [ ]:
from src.models.baselines import VAEOnlyBaseline
from torch.utils.data import DataLoader, TensorDataset

input_dim = raw_txn_features.shape[1]
vae_only = VAEOnlyBaseline(input_dim=input_dim, hidden=64, latent=16).to(device)
opt_vae = torch.optim.Adam(vae_only.parameters(), lr=0.001)

# Train on normal (non-fraud) transactions only
normal_mask = train_mask & (y == 0)
normal_data = raw_txn_features[normal_mask]
train_loader = DataLoader(TensorDataset(normal_data), batch_size=2048, shuffle=True)

print(f'Training VAE on {len(normal_data):,} normal transactions...')

t0 = time.time()
for epoch in range(1, 51):
    vae_only.train()
    total_loss = 0
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        opt_vae.zero_grad()
        # VAE forward returns reconstruction error
        recon_err = vae_only(batch_x)
        loss = recon_err.mean()  # minimize reconstruction error on normal data
        loss.backward()
        opt_vae.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d} | loss: {total_loss/len(train_loader):.4f}')

vae_time = time.time() - t0

# Score test set — higher reconstruction error = more suspicious
vae_only.eval()
with torch.no_grad():
    test_features = raw_txn_features[test_mask].to(device)
    vae_raw_scores = vae_only(test_features).cpu().numpy()

# Normalize scores to [0, 1] range for fair comparison
vae_scores = (vae_raw_scores - vae_raw_scores.min()) / (vae_raw_scores.max() - vae_raw_scores.min() + 1e-8)

vae_metrics = compute_metrics(y_test, vae_scores)
print(f'\nVAE-only trained in {vae_time:.1f}s')
print_report(y_test, vae_scores)

## 6. Model E — Hybrid GAT+VAE (Our Model)

This is our full model: graph attention + VAE anomaly detection combined.  
Two-phase training: first pre-train VAE, then train everything together.

In [ ]:
from src.models.hybrid_model import HybridGATVAE
from src.training.losses import focal_loss, reconstruction_loss, kl_divergence
import yaml

# Load config
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

in_channels = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}
raw_dim = raw_txn_features.shape[1]

model = HybridGATVAE(
    metadata=data.metadata(),
    in_channels=in_channels,
    raw_txn_dim=raw_dim,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
).to(device)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# --- Phase 1: Pre-train VAE on normal transactions ---
print('\n--- Phase 1: VAE pre-training ---')

# Freeze GAT and classifier, only train VAE
for p in model.gat.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = False

vae_opt = torch.optim.Adam(model.vae.parameters(), lr=0.001)
normal_loader = DataLoader(TensorDataset(normal_data), batch_size=2048, shuffle=True)

for epoch in range(1, 51):
    model.train()
    total_loss = 0
    for (batch_x,) in normal_loader:
        batch_x = batch_x.to(device)
        vae_opt.zero_grad()
        x_recon, mu, logvar = model.vae(batch_x)
        loss = reconstruction_loss(x_recon, batch_x) + 0.01 * kl_divergence(mu, logvar)
        loss.backward()
        vae_opt.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d} | loss: {total_loss/len(normal_loader):.4f}')

# Unfreeze everything
for p in model.parameters(): p.requires_grad = True
print('Phase 1 done.')

In [ ]:
# --- Phase 2: End-to-end training with differential LR ---
print('--- Phase 2: End-to-end training ---')

lr = cfg['training']['lr']
optimizer = torch.optim.Adam([
    {'params': model.gat.parameters(), 'lr': lr},
    {'params': model.vae.parameters(), 'lr': lr * 0.1},   # slow LR to keep VAE stable
    {'params': model.recon_bn.parameters(), 'lr': lr},
    {'params': model.classifier.parameters(), 'lr': lr * 2},  # fast LR for classifier
])

lambda1 = cfg['training']['lambda1']
lambda2 = cfg['training']['lambda2']

# Use NeighborLoader for mini-batch training on the graph
train_loader = NeighborLoader(
    data, num_neighbors=[5, 3], batch_size=4096,
    input_nodes=('transaction', train_mask), shuffle=True,
)

best_val_loss = float('inf')
patience_cnt = 0

t0 = time.time()
for epoch in range(1, cfg['training']['epochs'] + 1):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        bs = batch['transaction'].batch_size
        raw_txn = batch['transaction'].x

        optimizer.zero_grad()
        out = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)

        # Combined loss: focal + reconstruction + KL
        loss_cls = focal_loss(out['logit'][:bs], batch['transaction'].y[:bs].float())
        loss_rec = reconstruction_loss(out['x_recon'][:bs], raw_txn[:bs])
        loss_kl = kl_divergence(out['mu'][:bs], out['logvar'][:bs])
        loss = loss_cls + lambda1 * loss_rec + lambda2 * loss_kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    if epoch % 5 == 0:
        print(f'  Epoch {epoch:3d} | loss: {avg_loss:.4f}')

    # Simple early stopping on training loss
    if avg_loss < best_val_loss:
        best_val_loss = avg_loss
        patience_cnt = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_cnt += 1
        if patience_cnt >= 10:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)
hybrid_time = time.time() - t0
print(f'Phase 2 done in {hybrid_time:.1f}s')

In [ ]:
# Evaluate hybrid model on test set
model.eval()
model = model.to(device)

test_loader = NeighborLoader(
    data, num_neighbors=[-1, -1], batch_size=4096,
    input_nodes=('transaction', test_mask), shuffle=False,
)

all_scores, all_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        bs = batch['transaction'].batch_size
        raw_txn = batch['transaction'].x
        out = model(batch.x_dict, batch.edge_index_dict, raw_txn_features=raw_txn)
        all_scores.append(out['fraud_score'][:bs].cpu())
        all_targets.append(batch['transaction'].y[:bs].cpu())

hybrid_scores = torch.cat(all_scores).numpy()
hybrid_y = torch.cat(all_targets).numpy()

hybrid_metrics = compute_metrics(hybrid_y, hybrid_scores)
print_report(hybrid_y, hybrid_scores)

## 7. Ablation Results — All Models Side by Side

In [ ]:
# Build results table
results = pd.DataFrame({
    'Model': ['XGBoost', 'GCN Baseline', 'GAT-only', 'VAE-only', 'Hybrid GAT+VAE'],
    'F1 (Fraud)': [xgb_metrics['f1_fraud'], gcn_metrics['f1_fraud'], gat_metrics['f1_fraud'], vae_metrics['f1_fraud'], hybrid_metrics['f1_fraud']],
    'Recall': [xgb_metrics['recall_fraud'], gcn_metrics['recall_fraud'], gat_metrics['recall_fraud'], vae_metrics['recall_fraud'], hybrid_metrics['recall_fraud']],
    'Precision': [xgb_metrics['precision_fraud'], gcn_metrics['precision_fraud'], gat_metrics['precision_fraud'], vae_metrics['precision_fraud'], hybrid_metrics['precision_fraud']],
    'AUC-ROC': [xgb_metrics['roc_auc'], gcn_metrics['roc_auc'], gat_metrics['roc_auc'], vae_metrics['roc_auc'], hybrid_metrics['roc_auc']],
    'Avg Precision': [xgb_metrics['avg_precision'], gcn_metrics['avg_precision'], gat_metrics['avg_precision'], vae_metrics['avg_precision'], hybrid_metrics['avg_precision']],
})

print(results.to_string(index=False))

# Save
os.makedirs('../results/tables', exist_ok=True)
results.to_csv('../results/tables/ablation_results.csv', index=False)
print('\nSaved to results/tables/ablation_results.csv')

In [ ]:
# Bar chart comparing all models
metrics_to_plot = ['F1 (Fraud)', 'Recall', 'Precision', 'AUC-ROC']
x = np.arange(len(metrics_to_plot))
width = 0.15
colors = ['#2ECC71', '#5DADE2', '#F39C12', '#9B59B6', '#E74C3C']
models = results['Model'].tolist()

fig, ax = plt.subplots(figsize=(14, 6))
for i, model_name in enumerate(models):
    vals = [results.loc[i, m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=model_name, color=colors[i])
    # Put value on top of each bar
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics_to_plot, fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Ablation Study — What Each Component Adds', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/ablation_study.png', dpi=150)
plt.show()

## 8. What Does Each Component Add?

Let's answer the key questions:
- **GCN → GAT**: Does attention help over simple message passing?
- **GAT → Hybrid**: Does the VAE branch add value?
- **XGBoost vs Hybrid**: Does graph structure catch things tabular models miss?

In [ ]:
# Component contribution analysis
print('=== Component Contribution ===\n')

# Attention: GAT vs GCN
attn_gain = gat_metrics['f1_fraud'] - gcn_metrics['f1_fraud']
print(f'1. Attention mechanism (GAT vs GCN):')
print(f'   F1 gain: {attn_gain:+.4f}')
print(f'   -> {"Attention helps!" if attn_gain > 0 else "No improvement from attention"}')

# VAE branch: Hybrid vs GAT-only
vae_gain = hybrid_metrics['f1_fraud'] - gat_metrics['f1_fraud']
print(f'\n2. VAE branch (Hybrid vs GAT-only):')
print(f'   F1 gain: {vae_gain:+.4f}')
print(f'   -> {"VAE adds value!" if vae_gain > 0 else "VAE did not help"}')

# Graph structure: Hybrid vs XGBoost
graph_gain = hybrid_metrics['recall_fraud'] - xgb_metrics['recall_fraud']
print(f'\n3. Graph structure (Hybrid vs XGBoost):')
print(f'   Recall gain: {graph_gain:+.4f}')
print(f'   -> {"Graph catches more fraud!" if graph_gain > 0 else "XGBoost has better recall"}')

# Complementarity: what XGBoost misses that Hybrid catches
xgb_thresh = xgb_metrics['threshold']
hybrid_thresh = hybrid_metrics['threshold']
xgb_pred = (xgb_scores >= xgb_thresh).astype(int)
hybrid_pred = (hybrid_scores >= hybrid_thresh).astype(int)

only_hybrid = ((hybrid_pred == 1) & (xgb_pred == 0) & (y_test == 1)).sum()
only_xgb = ((hybrid_pred == 0) & (xgb_pred == 1) & (y_test == 1)).sum()
total_fraud = (y_test == 1).sum()

print(f'\n4. Complementarity:')
print(f'   Total fraud in test set: {total_fraud}')
print(f'   Only Hybrid catches: {only_hybrid} ({100*only_hybrid/total_fraud:.1f}%)')
print(f'   Only XGBoost catches: {only_xgb} ({100*only_xgb/total_fraud:.1f}%)')
print(f'   -> Hybrid finds {only_hybrid} fraud cases invisible to XGBoost')

In [ ]:
# Save the trained hybrid model
os.makedirs('../results/models', exist_ok=True)
torch.save(model.state_dict(), '../results/models/hybrid_gatvae_v2.pt')
print('Model saved: results/models/hybrid_gatvae_v2.pt')

print(f'\n=== Training Times ===')
print(f'  XGBoost:     {xgb_time:6.1f}s')
print(f'  GCN:         {gcn_time:6.1f}s')
print(f'  GAT-only:    {gat_time:6.1f}s')
print(f'  VAE-only:    {vae_time:6.1f}s')
print(f'  Hybrid:      {hybrid_time:6.1f}s')